# **CODE_MONKEYS_BOXERS_DB**

This is a program to take boxing data, store it in a SQLite database and manipulate it.

## Requirements:
- Have the data.csv with the header: `name;total_fights;wins;loses;draws` and have all data separated in that exact format.

In [46]:
import pandas as pd
import sqlite3

connection = sqlite3.connect("boxers.db")
cursor = connection.cursor()

In [47]:
def read_file(path: str) -> pd.DataFrame:
  """
	Read a file with values separated by commas or dashes and store it in a variable.

	Args:
		path (str): Path of the file (e.g. path/to/the/file).
	Returns:
		pd.DataFrame: Pandas object with all csv data.
  """
  try:
    dataframe = pd.read_csv(
      filepath_or_buffer=path,
      sep=";"
    )

    print("File read successfully")
    return dataframe
  except FileNotFoundError:
    print("Error: The file could not be loaded")
  except Exception as error:
    print("Unexpected error -> ", error)

read_file("data.csv")

File read successfully


,name,total_fights,wins,loses,draws
0,Mike Tyson,59,50,7,2
1,Roy Jones Jr.,76,66,10,0
2,Sugar Ray Leonard,40,36,3,1
3,Julio Cesar Chavez,115,107,6,2
4,Muhammad Ali,61,56,5,0
5,Roberto Duran,119,103,16,0
6,Evander Holyfield,57,44,10,2
7,Floyd Mayweather Jr.,50,50,0,0
8,Anthony Joshua,34,30,4,0
9,Oleksander Usyk,25,25,0,0


In [48]:
def create_database() -> None:
  """
    Creates the database of the boxers with the structure of id, name, total_matches, won, lost, draw.

    Returns:
      None: Shows to the user that the file has been created successfully.
  """
  try:
    cursor.execute("""
      CREATE TABLE boxers(
        id INTEGER PRIMARY KEY,
        name TEXT,
        total_fights INTEGER,
        wins INTEGER,
        loses INTEGER,
        draws INTEGER
      )
    """)
    print("Database created successfully")
  except Exception as error:
    print("Unexpected error -> ", error)

create_database()

Database created successfully


In [49]:
def populate_table(dataframe: pd.DataFrame) -> None:
  """
  Inserts all data from dataframe to the boxers table to populate it.

  Args:
    dataframe (pd.DataFrame): Pandas object that has all csv data.

  Returns:
    None: Shows to the user that the table has been populated successfully.
  """
  try:
    dataframe.to_sql("boxers", con=connection, if_exists='append', index_label="id")
    print("The table has been populated successfully")
  except Exception as error:
    print("Unexpected error -> ", error)

dataframe = read_file("data.csv")
populate_table(dataframe)

File read successfully
The table has been populated successfully


# Read Database Module

This notebook implements the `read_database` function to interact with the boxers SQLite database. It allows filtering and searching boxers dynamically.

In [ ]:
from typing import Optional, Dict, Any

def read_database(id: Optional[int] = None, name: Optional[str] = None) -> Dict[int, Dict[str, Any]]:
    """
    Retrieve boxer records from the SQLite database with optional filters.

    Queries the 'boxers' table. Filters by exact ID if supplied, or searches
    by partial name (case-insensitive) if specified. If no filters are provided,
    it retrieves all boxer records.

    Args:
        id (Optional[int]): The unique identifier of the boxer. Defaults to None.
        name (Optional[str]): Partial name of the boxer to search for. Defaults to None.

    Returns:
        Dict[int, Dict[str, Any]]: A dictionary mapped by boxer ID containing boxer details.
    """
    # Connect to the project's database
    connection = sqlite3.connect("boxers.db")
    cursor = connection.cursor()

    # Dynamically build the SQL query and its parameters
    query = "SELECT id, name, total_fights, wins, loses, draws FROM boxers"
    params = []
    conditions = []

    if id is not None:
        conditions.append("id = ?")
        params.append(id)
    if name is not None:
        conditions.append("name LIKE ?")
        params.append(f"%{name}%")

    if conditions:
        query += " WHERE " + " AND ".join(conditions)

    cursor.execute(query, params)
    rows = cursor.fetchall()
    connection.close()

    # Map results into a dictionary matching project specifications
    boxers_dict = {}
    for row in rows:
        boxer_id = row[0]
        boxers_dict[boxer_id] = {
            "name": row[1],
            "total_fights": row[2],
            "wins": row[3],
            "loses": row[4],
            "draws": row[5]
        }

    # Print formatted output to the terminal in English
    if not boxers_dict:
        print("No boxers found matching the search criteria.")
    else:
        print(f"{'ID':<5} | {'Name':<25} | {'Total':<6} | {'Wins':<9} | {'Loses':<9} | {'Draws':<7}")
        print("-" * 75)
        for b_id, data in boxers_dict.items():
            print(f"{b_id:<5} | {data['name']:<25} | {data['total_fights']:<6} | {data['wins']:<9} | {data['loses']:<9} | {data['draws']:<7}")

    return boxers_dict
